# LLM Grammar Experiment - Colab Runner
Make sure you're using a **GPU runtime**: Runtime > Change runtime type > T4 GPU

In [ ]:
# Step 1: Check GPU is available
!nvidia-smi

In [ ]:
# Step 2: Clone your repo
!git clone https://github.com/molybrine/Bsc-project.git
%cd Bsc-project
!git checkout master

In [ ]:
# Step 3: Install dependencies
!pip install -q torch transformers accelerate bitsandbytes sentencepiece
!pip install -q numpy pandas python-Levenshtein scipy statsmodels pingouin matplotlib seaborn tqdm pyyaml

In [ ]:
# Step 4: Verify GPU and VRAM
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM: {total:.1f} GB")

In [ ]:
# Step 5: Run bloomz at 8-bit
!python run_experiment.py --model bloomz --quantize 8bit

In [ ]:
# Step 6: Run pythia at 8-bit
!python run_experiment.py --model pythia --quantize 8bit

In [ ]:
# Step 7: Merge results and run full analysis (stats + figures)
import glob
bloomz_csv = sorted(glob.glob('results/results_bloomz_*.csv'))[-1]
pythia_csv = sorted(glob.glob('results/results_pythia_*.csv'))[-1]
print(f"Merging:\n  {pythia_csv}\n  {bloomz_csv}")
!python merge_and_analyse.py --model1 "{pythia_csv}" --model2 "{bloomz_csv}"

In [ ]:
# Step 8: View results summary
import pandas as pd
import glob

for f in sorted(glob.glob('results/*.csv')):
    df = pd.read_csv(f)
    print(f"\n{'='*50}")
    print(f"File: {f}")
    print(f"Instances: {len(df)}")
    print(f"Overall accuracy: {df['exact_match'].mean():.3f}")
    print(f"Mean edit distance: {df['edit_distance'].mean():.3f}")
    print(f"\nAccuracy by variant:")
    print(df.groupby('variant')['exact_match'].mean())
    print(f"\nAccuracy by shot count:")
    print(df.groupby('shot_count')['exact_match'].mean())

In [ ]:
# Step 9: Display generated figures
from IPython.display import Image, display
import os

figures = ['learning_curves.png', 'interaction_plot.png', 'heatmap.png', 'error_analysis.png']
for fig in figures:
    path = f'figures/{fig}'
    if os.path.exists(path):
        print(f"\n{'='*50}\n{fig}\n{'='*50}")
        display(Image(filename=path))
    else:
        print(f"Missing: {path}")

In [ ]:
# Step 10: Custom figure — Pythia at 8-shot (interaction plot)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('results/results_merged.csv')

# Add factor columns if not present
if 'syntax' not in df.columns:
    df['syntax'] = df['variant'].map({'v1': 'SVO', 'v2': 'VSO', 'v3': 'SVO', 'v4': 'VSO'})
if 'morphology' not in df.columns:
    df['morphology'] = df['variant'].map({'v1': 'none', 'v2': 'none', 'v3': 'case', 'v4': 'case'})

# Filter to Pythia at 8-shot only
pythia_8 = df[(df['model'] == 'pythia') & (df['shot_count'] == 8)]

fig, ax = plt.subplots(figsize=(7, 5))
summary = pythia_8.groupby(['syntax', 'morphology'])['exact_match'].mean().reset_index()
sns.pointplot(
    data=summary,
    x='syntax', y='exact_match',
    hue='morphology', ax=ax,
    markers=['o', 's'],
    palette=['#2E86C1', '#E74C3C'],
)
ax.set_ylabel('Exact Match Accuracy')
ax.set_title('Pythia (8-shot): Syntax x Morphology Interaction')
plt.tight_layout()
plt.savefig('figures/pythia_8shot_interaction.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: figures/pythia_8shot_interaction.png")

In [ ]:
# Step 11: Custom figure — Error analysis at 8-shot (both models side by side)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('results/results_merged.csv')

# Filter to 8-shot only, then group by model x variant
shot8 = df[df['shot_count'] == 8]
err = shot8.groupby(['model', 'variant']).agg({
    'word_order_correct': 'mean',
    'case_marking_correct': 'mean',
}).reset_index()

# Sort so pythia appears first, then bloomz, variants in order
err = err.sort_values(['model', 'variant']).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(err))
width = 0.35
ax.bar(x - width / 2, err['word_order_correct'],
       width, label='Word Order', color='#2E86C1')
ax.bar(x + width / 2, err['case_marking_correct'],
       width, label='Case Marking', color='#E74C3C')
ax.set_xticks(x)
ax.set_xticklabels(
    [f"{r['model']}\n{r['variant']}" for _, r in err.iterrows()],
    fontsize=9,
)
ax.set_ylabel('Feature Accuracy')
ax.set_title('Error Analysis at 8-shot: Pythia vs BLOOMZ')
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig('figures/error_analysis_8shot.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: figures/error_analysis_8shot.png")

In [ ]:
# Step 12: Download results and figures
from google.colab import files
import glob

for f in glob.glob('results/*.csv'):
    files.download(f)
for f in glob.glob('figures/*.png'):
    files.download(f)